In [1]:
import torch
import matplotlib.pyplot as plt
import os

In [2]:
def generate_all_plots(models_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    # Model name to directory mapping
    model_configs = {
        "Model 1 (GloVe)":      "Model1",
        "Model 2 (GloVeAttn)":  "Model2",
        "Model 3 (FrozenBERT)": "Model3",
        "Model 4 (TunedBERT)":  "Model4"
    }
    
    histories = {}
    for label, folder in model_configs.items():
        hist_path = os.path.join(models_dir, folder, "training_histories.pt")
        if os.path.exists(hist_path):
            data = torch.load(hist_path, map_location='cpu')
            # Extract the actual history dictionary (first key)
            key = list(data.keys())[0]
            histories[label] = data[key]
            print(f"Loaded history for {label} from {hist_path}")
        else:
            print(f"Warning: {hist_path} not found.")

    if not histories:
        print("No histories loaded. Exiting.")
        return

    # Metrics to plot
    metrics = [
        ("train_losses", "training_loss.png",   "Training Loss",   "Loss"),
        ("val_losses",   "validation_loss.png", "Validation Loss", "Loss"),
        ("val_bleu",     "bleu_curve.png",      "Validation BLEU", "Score"),
        ("val_chrf",     "chrf_curve.png",      "Validation chrF", "Score"),
        ("val_ter",      "ter_curve.png",       "Validation TER",  "Score")
    ]
    
    # Use a professional style
    plt.style.use('seaborn-v0_8-muted') # fallback or consistent style
    
    for key, filename, title, ylabel in metrics:
        plt.figure(figsize=(10, 6))
        
        found_any = False
        for label, hist in histories.items():
            if key in hist and hist[key]:
                data_points = hist[key]
                epochs = range(1, len(data_points) + 1)
                plt.plot(epochs, data_points, label=label, marker='o', markersize=4, alpha=0.8)
                found_any = True
        
        if not found_any:
            print(f"No data found for metric {key}. Skipping {filename}.")
            plt.close()
            continue
            
        plt.title(title, fontsize=14, fontweight='bold')
        plt.xlabel("Epoch", fontsize=12)
        plt.ylabel(ylabel, fontsize=12)
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        
        save_path = os.path.join(output_dir, filename)
        plt.savefig(save_path, dpi=300)
        plt.close()
        print(f"Generated {save_path}")

In [ ]:
if __name__ == "__main__":
    # Assuming models are in Assignment2/models and we want them in 2022CS11621_A1_Part2/plots/
    # If run from root:
    MODELS_DIR = "models"
    OUTPUT_DIR = "2022CS11621_A1_Part2/plots"
    
    # If run inside the submission dir, we would adjust relative paths.
    # We'll make it flexible.
    if not os.path.exists(MODELS_DIR) and os.path.exists("../models"):
        MODELS_DIR = "../models"
    
    generate_all_plots(MODELS_DIR, OUTPUT_DIR)